# 05.7 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/05-powerful-models/07-imbalanced-data/exercises.ipynb)

Exercises 1 and 2 are on paper; check them in code afterwards.

## Exercise 1: the lying model (paper)

ShopWave has 20,000 orders and 60 are fraud. a) Accuracy of "nothing is fraud"? b) Its recall? c) PR-AUC of a random scorer? d) Class weights with `"balanced"`?

*Your answer here.*

## Exercise 2: SMOTE by hand (paper)

Rare point $x = (10, 2)$, nearest rare neighbour $x_{nn} = (14, 1)$. Compute the synthetic point for $\lambda = 0.1$, $0.5$ and $0.9$. Where do they lie?

*Your answer here.*

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail
from sklearn.model_selection import train_test_split

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

# New, rare target: the top 5% of customers by spend in Sep-Nov 2011
threshold_gbp = future_spend.quantile(0.95)
cust["big_spender"] = (future_spend > threshold_gbp).astype(int)
Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    cust[features], cust["big_spender"], test_size=0.25, random_state=42, stratify=cust["big_spender"])
print(f"big spender = more than £{threshold_gbp:,.0f} next quarter")
print(f"train: {yb_train.sum()} of {len(yb_train):,} ({yb_train.mean():.1%})   "
      f"test: {yb_test.sum()} of {len(yb_test):,} ({yb_test.mean():.1%})")

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score)

def report(name, y_true, proba, t=0.5):
    pred = (proba >= t).astype(int)
    return {"model": name, "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred), "F1": f1_score(y_true, pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_true, proba), "PR-AUC": average_precision_score(y_true, proba)}

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.pipeline import make_pipeline as imb_pipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

LR = lambda **kw: LogisticRegression(max_iter=1000, **kw)
models = {
    "plain":              make_pipeline(StandardScaler(), LR()),
    "class_weight":       make_pipeline(StandardScaler(), LR(class_weight="balanced")),
    "random oversample":  imb_pipeline(StandardScaler(), RandomOverSampler(random_state=42), LR()),
    "random undersample": imb_pipeline(StandardScaler(), RandomUnderSampler(random_state=42), LR()),
    "SMOTE":              imb_pipeline(StandardScaler(), SMOTE(random_state=42), LR()),
}

## Exercise 3: a rarer target

Change the big-spender cut to the top **2%** (`quantile(0.98)`). Rebuild the split and the comparison table. What happens to the plain model's recall at 0.5 and to PR-AUC?

In [ ]:
# your code here

## Exercise 4: imbalance with a tree model

Repeat the comparison with `lgb.LGBMClassifier(n_estimators=200, learning_rate=0.03, num_leaves=8, verbose=-1, n_jobs=1)` instead of logistic regression: plain, `class_weight="balanced"`, SMOTE in an imblearn pipeline. Which helps most? Does PR-AUC change?

In [ ]:
# your code here